In [100]:
import glob
import shutil

import numpy as np
import openml
import optuna
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
import torch
from torch.utils.data import DataLoader
from torch.utils.data.dataset import TensorDataset


In [101]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [102]:
CLASS_LABELS = ["walking", "upstairs", "downstairs", "sitting", "standing", "laying"]

dataset = openml.datasets.get_dataset(1478)
X, y, *_ = dataset.get_data(target=dataset.default_target_attribute)
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Needed as separate array for KFold split
y_train_labels = y_train.astype(int) - 1  # Converts "5" -> 4

X_train = torch.tensor(X_train.to_numpy(), dtype=torch.float32, device=device)
y_train = torch.tensor(pd.get_dummies(y_train).to_numpy(), dtype=torch.float32, device=device)

X_train.shape, y_train.shape

(torch.Size([8239, 561]), torch.Size([8239, 6]))

In [103]:
class MLP(nn.Module):
    def __init__(self, n_hidden: int, n_width: int):
        super().__init__()

        self.input_layer = nn.Linear(X.shape[1], n_width)
        self.hidden_layers = nn.ModuleList([nn.Linear(n_width, n_width) for _ in range(n_hidden)])
        self.output_layer = nn.Linear(n_width, len(CLASS_LABELS))
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = self.relu(layer(x))
        x = self.output_layer(x)
        return x


In [104]:
def train_one_epoch(model, dataloader, optimizer, loss_fn) -> float:
    model.train()
    train_loss = 0.0

    for data, target in dataloader:
        optimizer.zero_grad()

        y_pred = model(data)

        loss = loss_fn(y_pred, target)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

    train_loss /= len(dataloader)
    return train_loss

In [105]:
def eval_model(model, dataloader, loss_fn) -> float:
    model.eval()
    with torch.no_grad():
        val_loss = 0
        for data, target in dataloader:
            y_pred = model(data)
            val_loss += loss_fn(y_pred, target).item()

    val_loss /= len(dataloader)
    return val_loss

In [106]:
def train_model(fold: int, model, train_loader, val_loader, optimizer, loss_fn, epochs: int):
    best_val_loss = 999999999.9

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
        val_loss = eval_model(model, val_loader, loss_fn)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(model.state_dict(), f"../checkpoints/fold_{fold}.pth")

In [107]:
def objective_mlp(trial: optuna.trial.Trial) -> float:
    n_hidden: int = trial.suggest_int("n_hidden", 1, 10)
    n_width: int = trial.suggest_int("n_width", 10, 300, log=True)
    lr: float = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    batch_size: int = trial.suggest_categorical("batch_size", (2**np.arange(5, 11)).tolist())

    skf = StratifiedKFold(n_splits=5, shuffle=True)

    total_val_loss = 0.0

    current_params = trial.params
    print(f"{current_params=}")
    torch.save(current_params, f"../checkpoints/model_params.pth")

    for fold, (train_index, val_index) in enumerate(skf.split(X_train, y_train_labels)):
        model = MLP(n_hidden, n_width)
        model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        train_ds = TensorDataset(X_train[train_index], y_train[train_index])
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

        val_ds = TensorDataset(X_train[val_index], y_train[val_index])
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True)

        train_model(fold, model, train_loader, val_loader, optimizer, loss_fn, epochs=100)

        model.load_state_dict(torch.load(f"../checkpoints/fold_{fold}.pth"))
        total_val_loss += eval_model(model, val_loader, loss_fn)

    avg_val_loss = total_val_loss / 5
    return avg_val_loss


In [108]:
def save_best_models_callback(study, trial):
    if study.best_trial.number == trial.number:
        # The best trial yet
        for file_path in glob.glob("../checkpoints/*.pth"):
            shutil.copy(file_path, "../checkpoints/best/")

In [109]:
study = optuna.create_study(study_name="Study", direction='minimize')
study.optimize(objective_mlp, n_trials=30, callbacks=[save_best_models_callback])

[I 2026-04-26 23:15:17,054] A new study created in memory with name: Study


current_params={'n_hidden': 2, 'n_width': 48, 'lr': 0.0034128722233636543, 'batch_size': 32}


[I 2026-04-26 23:16:11,938] Trial 0 finished with value: 0.04672880012591262 and parameters: {'n_hidden': 2, 'n_width': 48, 'lr': 0.0034128722233636543, 'batch_size': 32}. Best is trial 0 with value: 0.04672880012591262.


current_params={'n_hidden': 5, 'n_width': 51, 'lr': 0.005127930637169973, 'batch_size': 32}


[I 2026-04-26 23:17:26,883] Trial 1 finished with value: 0.05153349292489181 and parameters: {'n_hidden': 5, 'n_width': 51, 'lr': 0.005127930637169973, 'batch_size': 32}. Best is trial 0 with value: 0.04672880012591262.


...

[I 2026-04-26 23:36:52,461] Trial 27 finished with value: 0.04545778387154524 and parameters: {'n_hidden': 4, 'n_width': 64, 'lr': 0.000929369410632861, 'batch_size': 128}. Best is trial 13 with value: 0.0396398393382137.


current_params={'n_hidden': 1, 'n_width': 128, 'lr': 0.00033650150088507935, 'batch_size': 512}


[I 2026-04-26 23:37:08,368] Trial 28 finished with value: 0.058776233531534675 and parameters: {'n_hidden': 1, 'n_width': 128, 'lr': 0.00033650150088507935, 'batch_size': 512}. Best is trial 13 with value: 0.0396398393382137.


current_params={'n_hidden': 2, 'n_width': 61, 'lr': 0.004000906920447919, 'batch_size': 32}


[I 2026-04-26 23:38:05,203] Trial 29 finished with value: 0.046673533182159256 and parameters: {'n_hidden': 2, 'n_width': 61, 'lr': 0.004000906920447919, 'batch_size': 32}. Best is trial 13 with value: 0.0396398393382137.


In [110]:
study.best_params

{'n_hidden': 1, 'n_width': 98, 'lr': 0.0007726630559224028, 'batch_size': 32}